# DVAD - checkpoint-500 eval on the real public test set

EVAL ONLY - no training. Loads the base Qwen2.5-VL-3B (4-bit) + the
checkpoint-500 LoRA adapter (from the interrupted fine-tune run) and runs
inference on 33 real test-video frames, so this can be scored macro-F1
against ground_truth.csv exactly like the classifier was.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import subprocess, sys
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())

In [ ]:
!pip install -q --upgrade unsloth unsloth_zoo
import torch
assert torch.cuda.is_available(), "No GPU. Set Accelerator to GPU in session options."
print("gpu:", torch.cuda.get_device_name(0), "cc:", torch.cuda.get_device_capability(0))

In [ ]:
# Locate the two attached datasets without hardcoding mount depth.
import glob, json
from pathlib import Path

ckpt_candidates = sorted(glob.glob("/kaggle/input/**/adapter_config.json", recursive=True))
eval_candidates = sorted(glob.glob("/kaggle/input/**/eval.jsonl", recursive=True))
assert ckpt_candidates, "No adapter_config.json found - attach the dvad-vlm-ckpt500 dataset."
assert eval_candidates, "No eval.jsonl found - attach the dvad-vlm-eval-test dataset."
CKPT_DIR = Path(ckpt_candidates[0]).parent
EVAL_ROOT = Path(eval_candidates[0]).parent
print("checkpoint:", CKPT_DIR)
print("eval data :", EVAL_ROOT)

rows = [json.loads(l) for l in (EVAL_ROOT / "eval.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
print(f"{len(rows)} test video frames to evaluate")

In [ ]:
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen2.5-VL-3B-Instruct",
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth",
)
print("base model loaded")

from peft import PeftModel
model = PeftModel.from_pretrained(model, str(CKPT_DIR))
print("checkpoint-500 adapter attached")
FastVisionModel.for_inference(model)

In [ ]:
INSTRUCTION = (
    "You are a drone and CCTV video anomaly analyst. Look at this frame from a "
    "traffic or public-space camera and decide whether it shows an anomaly that "
    "an operator must respond to.\n"
    "Answer with JSON only, using exactly these keys:\n"
    '{"is_anomaly": true|false, "class_name": "<one of: normal, traffic_accident, '
    "traffic_congestion, stalled_or_broken_down_vehicle, vehicle_blocking_traffic, "
    "wrong_way_driving, road_spill_or_debris, waterlogging_or_flood, fire, smoke, "
    'fighting_or_violence, loitering_or_suspicious_presence>", '
    '"description_summary": "<one short sentence describing what you see>"}'
)

def predict(image_path, max_new_tokens=200):
    """Same explicit-kwargs + inference_mode + autocast pattern proven during
    training-time eval - positional image/text args on a 4-bit T4 previously
    produced NaN logits ("!!!!!!!" repeated), not a bad fine-tune."""
    img = Image.open(image_path).convert("RGB")
    messages = [
        {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": INSTRUCTION}]},
    ]
    text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(text=[text], images=[img], return_tensors="pt", padding=True).to("cuda")
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             use_cache=True, pad_token_id=tokenizer.tokenizer.eos_token_id)
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

from PIL import Image
import torch, time, json as _json

# MULTI-FRAME. The previous single-midpoint-frame eval missed every long
# video it got wrong (T025/T026/T028/T031, all 240s+) - one frame out of
# 6000-18846 cannot see an accident at second 30 of a 4-minute clip. Frames
# per video now scale with duration (3 for short clips, up to 14 for the
# 10-minute ones), and a video counts as anomalous if ANY frame sees
# something: a real event is transient, so demanding agreement across all
# frames would defeat the purpose of sampling more of them.
per_frame = []
t0 = time.time()
for i, r in enumerate(rows):
    img_path = EVAL_ROOT / r["image"]
    raw = predict(img_path)
    parsed = None
    try:
        parsed = _json.loads(raw[raw.index("{"): raw.rindex("}") + 1])
    except Exception:
        parsed = None
    per_frame.append({"video_id": r["video_id"], "frame_idx": r.get("frame_idx"),
                      "t_sec": r.get("t_sec"), "raw": raw[:300], "parsed": parsed})
    if (i + 1) % 20 == 0 or i == 0:
        lab = parsed.get("class_name") if parsed else "UNPARSEABLE"
        print(f"[{i+1}/{len(rows)}] {r['video_id']} t={r.get('t_sec')}s -> {lab}")

elapsed = time.time() - t0
print(f"\ndone in {elapsed/60:.1f} min, {elapsed/max(len(rows),1):.1f}s/frame")

with open("/kaggle/working/eval_frames.jsonl", "w", encoding="utf-8") as f:
    for row in per_frame:
        f.write(_json.dumps(row) + "\n")

from collections import defaultdict, Counter
by_video = defaultdict(list)
for r in per_frame:
    by_video[r["video_id"]].append(r)

results = []
for vid, frames in by_video.items():
    labels = [f["parsed"]["class_name"] for f in frames
              if f["parsed"] and f["parsed"].get("class_name")]
    anomalies = [l for l in labels if l != "normal"]
    if anomalies:
        # Most frequent anomaly wins: a real event usually persists across
        # several sampled frames, while a one-off misfire does not.
        label = Counter(anomalies).most_common(1)[0][0]
        best = next(f for f in frames
                    if f["parsed"] and f["parsed"].get("class_name") == label)
        desc = best["parsed"].get("description_summary", "")
        is_anom = True
    else:
        label, desc, is_anom = "normal", "", False
    results.append({"video_id": vid, "class_name": label, "is_anomaly": is_anom,
                    "description_summary": desc, "n_frames": len(frames),
                    "n_anomaly_frames": len(anomalies),
                    "votes": dict(Counter(labels))})
    print(f"{vid:<8} {len(anomalies):2d}/{len(frames):2d} anomaly frames -> {label}")

with open("/kaggle/working/eval_results.jsonl", "w", encoding="utf-8") as f:
    for row in results:
        f.write(_json.dumps(row) + "\n")

parseable = sum(1 for r in per_frame if r["parsed"] is not None)
print(f"\nframe-level parseable: {parseable}/{len(per_frame)}")
print(f"videos with >=1 anomaly frame: "
      f"{sum(1 for r in results if r['is_anomaly'])}/{len(results)}")

## Download the results

    kaggle kernels output <owner>/<slug> -p C:\dvad\models\kaggle_eval_output

Then locally: build a predictions.csv from eval_results.jsonl and score it
with score_submission.py against the real ground_truth.csv - the same
apples-to-apples comparison used for the classifier.